In [1]:
import pandas as pd
import numpy as np

def load_dataset(path: str) -> pd.DataFrame:
    """
    Loads the car dataset from a CSV file.

    Parameters
    ----------
    path : str
        Path to the CSV file.

    Returns
    -------
    pd.DataFrame
        Loaded dataset as a pandas DataFrame.
    """
    df = pd.read_csv(path)
    return df


# Load dataset
df = load_dataset("CarsDatasets.csv")

# Display first rows
df.head()


,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


In [2]:
def inspect_dataset(df: pd.DataFrame):
    """
    Prints basic structural information about the dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset.
    """
    print("Shape:", df.shape)
    print("\nColumn types:\n")
    print(df.dtypes)
    print("\nMissing values per column:\n")
    print(df.isnull().sum())


inspect_dataset(df)


Shape: (1218, 11)

Column types:

Company Names                object
Cars Names                   object
Engines                      object
CC/Battery Capacity          object
HorsePower                   object
Total Speed                  object
Performance(0 - 100 )KM/H    object
Cars Prices                  object
Fuel Types                   object
Seats                         int64
Torque                       object
dtype: object

Missing values per column:

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          3
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  0
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64


In [3]:
def split_features_target(df: pd.DataFrame, target_col: str):
    """
    Splits dataset into features X and target y.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset.
    target_col : str
        Name of target column.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix.
    y : pd.Series
        Target vector.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y


X_raw, y_raw = split_features_target(df, "Cars Prices")

print("Features shape:", X_raw.shape)
print("Target shape:", y_raw.shape)


Features shape: (1218, 10)
Target shape: (1218,)


In [4]:
def identify_feature_types(X: pd.DataFrame):
    """
    Identifies categorical and numerical columns based on data type.

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix.

    Returns
    -------
    categorical_cols : list
        List of categorical column names.
    numerical_cols : list
        List of numerical column names.
    """
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()
    return categorical_cols, numerical_cols


categorical_cols, numerical_cols = identify_feature_types(X_raw)

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)


Categorical: ['Company Names', 'Cars Names', 'Engines', 'CC/Battery Capacity', 'HorsePower', 'Total Speed', 'Performance(0 - 100 )KM/H', 'Fuel Types', 'Torque']
Numerical: ['Seats']


In [5]:
from sklearn.model_selection import train_test_split

def split_data(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42):
    """
    Splits data into train, validation, and test sets.

    Parameters
    ----------
    X : array-like
        Feature matrix.
    y : array-like
        Target vector.
    train_ratio : float
        Proportion of training data.
    val_ratio : float
        Proportion of validation data.
    random_state : int
        Random seed.

    Returns
    -------
    X_train, X_val, X_test, y_train, y_val, y_test
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(1 - train_ratio), random_state=random_state
    )

    val_size = val_ratio / (1 - train_ratio)

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_size), random_state=random_state
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(X_raw, y_raw)

print("Train:", X_train_raw.shape)
print("Val:", X_val_raw.shape)
print("Test:", X_test_raw.shape)


Train: (852, 10)
Val: (182, 10)
Test: (184, 10)


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_mlp(input_dim: int) -> tf.keras.Model:
    """
    Builds a baseline MLP model for regression.

    Parameters
    ----------
    input_dim : int
        Number of input features.

    Returns
    -------
    model : tf.keras.Model
        Compiled MLP model.
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="linear")
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    return model


# Example (will work after encoding)
# model = build_mlp(input_dim=NUM_FEATURES)
# model.summary()


In [7]:
categorical_cols, numerical_cols = identify_feature_types(X_train_raw)

print("Categorical columns:\n", categorical_cols)
print("Numerical columns:\n", numerical_cols)


Categorical columns:
 ['Company Names', 'Cars Names', 'Engines', 'CC/Battery Capacity', 'HorsePower', 'Total Speed', 'Performance(0 - 100 )KM/H', 'Fuel Types', 'Torque']
Numerical columns:
 ['Seats']


In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

def one_hot_preprocess(X_train, X_val, X_test, categorical_cols, numerical_cols):
    """
    Applies One-Hot Encoding to categorical features and standardization to numerical features.

    Returns processed train, validation, and test sets.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ]
    )

    X_train_enc = preprocessor.fit_transform(X_train)
    X_val_enc   = preprocessor.transform(X_val)
    X_test_enc  = preprocessor.transform(X_test)

    return X_train_enc, X_val_enc, X_test_enc, preprocessor


X_train_oh, X_val_oh, X_test_oh, oh_preprocessor = one_hot_preprocess(
    X_train_raw, X_val_raw, X_test_raw,
    categorical_cols, numerical_cols
)

print("One-Hot Encoded Train shape:", X_train_oh.shape)
print("One-Hot Encoded Val shape:", X_val_oh.shape)
print("One-Hot Encoded Test shape:", X_test_oh.shape)



One-Hot Encoded Train shape: (852, 2305)
One-Hot Encoded Val shape: (182, 2305)
One-Hot Encoded Test shape: (184, 2305)


In [9]:
import re
import numpy as np
import pandas as pd

def parse_price_column(y: pd.Series) -> pd.Series:
    """
    Converts heterogeneous price strings into numeric floats.

    Handles:
    - "$1,200,000"        -> 1200000
    - "12000-15000"      -> 13500
    - "55000 / 65000"    -> 60000
    - "35000"            -> 35000
    - Any mixed format   -> mean of all numbers found
    """

    def parse_single_value(val):
        if pd.isna(val):
            return np.nan

        val = str(val)

        # Extract all numbers from the string
        numbers = re.findall(r"\d+\.?\d*", val.replace(",", ""))

        if len(numbers) == 0:
            return np.nan

        numbers = [float(n) for n in numbers]

        # If multiple numbers exist, take their mean
        return np.mean(numbers)

    return y.apply(parse_single_value)


# 1. Parse target
y_raw = parse_price_column(df["Cars Prices"])
X_raw = df.drop(columns=["Cars Prices"])

# Sanity check
print("Parsed target sample:\n", y_raw.head())
print(y_raw.describe())
print(y_raw.dtype)
print("\nMissing values in target:", y_raw.isna().sum())



Parsed target sample:
 0    1100000.0
1     460000.0
2      13500.0
3     161000.0
4     253290.0
Name: Cars Prices, dtype: float64
count    1.217000e+03
mean     1.380370e+05
std      7.110424e+05
min      4.000000e+03
25%      2.800000e+04
50%      4.250000e+04
75%      7.000000e+04
max      1.800000e+07
Name: Cars Prices, dtype: float64
float64

Missing values in target: 1


In [10]:
from sklearn.preprocessing import LabelEncoder

def label_encode_features(X_train, X_val, X_test, categorical_cols):
    """
    Converts categorical features into integer indices for embedding layers.
    """
    X_train_le = X_train.copy()
    X_val_le   = X_val.copy()
    X_test_le  = X_test.copy()

    encoders = {}

    for col in categorical_cols:
        le = LabelEncoder()
        X_train_le[col] = le.fit_transform(X_train[col])
        X_val_le[col]   = le.transform(X_val[col])
        X_test_le[col]  = le.transform(X_test[col])
        encoders[col] = le

        print(f"{col:<25} | {len(le.classes_):<15}")

    return X_train_le, X_val_le, X_test_le, encoders


In [11]:
def build_mlp_with_embeddings(cat_dims, num_dim, emb_dim=8):
    """
    Builds an MLP model with embedding layers for categorical features.

    Parameters
    ----------
    cat_dims : dict
        Dictionary of {feature_name: number_of_unique_values}
    num_dim : int
        Number of numerical features
    emb_dim : int
        Size of embedding vectors

    Returns
    -------
    model : tf.keras.Model
    """
    inputs = []
    embeddings = []

    for name, dim in cat_dims.items():
        inp = layers.Input(shape=(1,), name=name)
        emb = layers.Embedding(input_dim=dim, output_dim=emb_dim)(inp)
        emb = layers.Flatten()(emb)
        inputs.append(inp)
        embeddings.append(emb)

    num_input = layers.Input(shape=(num_dim,), name="numerical")
    inputs.append(num_input)
    embeddings.append(num_input)

    x = layers.Concatenate()(embeddings)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    output = layers.Dense(1)(x)

    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    return model


In [12]:
# Step 1 – Identify categorical vs numerical

def identify_feature_types(X: pd.DataFrame):
    """
    Splits features into categorical and numerical columns.

    Categorical: object or category dtype
    Numerical: int or float dtype
    """
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    return categorical_cols, numerical_cols


In [16]:
# Step 2 – Target Encoding

def target_encode(X_train, X_val, X_test, y_train, categorical_cols):
    """
    Applies target encoding to categorical features.

    Each category is replaced by the mean target value
    computed only on the training set.
    """
    X_train_enc = X_train.copy()
    X_val_enc   = X_val.copy()
    X_test_enc  = X_test.copy()

    encoding_maps = {}
    global_mean = y_train.mean()

    for col in categorical_cols:
        means = y_train.groupby(X_train[col]).mean()
        encoding_maps[col] = means

        X_train_enc[col] = X_train[col].map(means)
        X_val_enc[col]   = X_val[col].map(means)
        X_test_enc[col]  = X_test[col].map(means)
        
        print(f"Encoded {col} with {len(means)} categories.")
        print(f"Global mean for unseen categories: {global_mean:.2f}")

        # fill unseen categories
        X_train_enc[col] = X_train_enc[col].fillna(global_mean)
        X_val_enc[col]   = X_val_enc[col].fillna(global_mean)
        X_test_enc[col]  = X_test_enc[col].fillna(global_mean)

    return X_train_enc, X_val_enc, X_test_enc, encoding_maps


In [17]:
# Step 3 – Standardization

def standardize_features(X_train, X_val, X_test, numerical_cols):
    """
    Standardizes numerical features using training statistics.

    x' = (x - mean) / std
    """
    X_train_std = X_train.copy()
    X_val_std   = X_val.copy()
    X_test_std  = X_test.copy()

    stats = {}

    for col in numerical_cols:
        mean = X_train[col].mean()
        std  = X_train[col].std()

        stats[col] = (mean, std)

        X_train_std[col] = (X_train[col] - mean) / std
        X_val_std[col]   = (X_val[col] - mean) / std
        X_test_std[col]  = (X_test[col] - mean) / std

    print("Standardization stats:")
    for col, (mean, std) in stats.items():
        print(f"  {col}: mean={mean:.2f}, std={std:.2f}")
        print("X_train shape:", X_train_std.shape)
    return X_train_std, X_val_std, X_test_std, stats


In [18]:
# Step 4 – Full Part 3 Pipeline

# 1. Parse target FIRST
y_raw = parse_price_column(df["Cars Prices"])
X_raw = df.drop(columns=["Cars Prices"])

# Sanity check (this should be float)
print("Target Type: ", y_raw.dtype)
print(y_raw.head())

# 2. Split
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(X_raw, y_raw)

# Sanity check again
print("Target Type: ", y_train.dtype)

# 3. Identify feature types
categorical_cols, numerical_cols = identify_feature_types(X_train_raw)

# 4. Target encode
print("Applying target encoding...")
X_train_te, X_val_te, X_test_te, te_maps = target_encode(
    X_train_raw, X_val_raw, X_test_raw, y_train, categorical_cols
)


# Standardize numerical features
X_train_final, X_val_final, X_test_final, scaling_stats = standardize_features(
    X_train_te, X_val_te, X_test_te, numerical_cols
)


Target Type:  float64
0    1100000.0
1     460000.0
2      13500.0
3     161000.0
4     253290.0
Name: Cars Prices, dtype: float64
Target Type:  float64
Applying target encoding...
Encoded Company Names with 35 categories.
Global mean for unseen categories: 162434.72
Encoded Cars Names with 846 categories.
Global mean for unseen categories: 162434.72
Encoded Engines with 282 categories.
Global mean for unseen categories: 162434.72
Encoded CC/Battery Capacity with 270 categories.
Global mean for unseen categories: 162434.72
Encoded HorsePower with 374 categories.
Global mean for unseen categories: 162434.72
Encoded Total Speed with 94 categories.
Global mean for unseen categories: 162434.72
Encoded Performance(0 - 100 )KM/H with 159 categories.
Global mean for unseen categories: 162434.72
Encoded Fuel Types with 18 categories.
Global mean for unseen categories: 162434.72
Encoded Torque with 223 categories.
Global mean for unseen categories: 162434.72
Standardization stats:
  Seats: mean

In [19]:
# 4.1 Activation Functions

def relu(x):
    """ReLU activation function."""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of ReLU."""
    return (x > 0).astype(float)


In [20]:
# 4.2 Loss Functions

def mse(y_true, y_pred):
    """Mean Squared Error loss."""
    return np.mean((y_true - y_pred) ** 2)

def mse_derivative(y_true, y_pred):
    """Derivative of MSE."""
    return 2 * (y_pred - y_true) / len(y_true)


In [22]:
# 4.3 MLP Class (Core of the Assignment)

class MLPRegressor:
    """
    A simple Multilayer Perceptron for regression.
    
    Architecture:
    Input -> Hidden1 -> Hidden2 -> Output
    """

    def __init__(self, input_dim, hidden1=64, hidden2=32, lr=0.001):
        self.lr = lr

        # Weight initialization
        self.W1 = np.random.randn(input_dim, hidden1) * 0.01
        self.b1 = np.zeros((1, hidden1))
        print("Initialized W1 with shape:", self.W1.shape)
        print("Initialized b1 with shape:", self.b1.shape)

        self.W2 = np.random.randn(hidden1, hidden2) * 0.01
        self.b2 = np.zeros((1, hidden2))
        print("\nInitialized W2 with shape:", self.W2.shape)
        print("Initialized b2 with shape:", self.b2.shape)

        self.W3 = np.random.randn(hidden2, 1) * 0.01
        self.b3 = np.zeros((1, 1))
        print("\nInitialized W3 with shape:", self.W3.shape)
        print("Initialized b3 with shape:", self.b3.shape)

    def forward(self, X):
        """Forward pass."""
        self.z1 = X @ self.W1 + self.b1
        self.a1 = relu(self.z1)
        print("\nAfter first layer: z1 shape:", self.z1.shape, "a1 shape:", self.a1.shape)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = relu(self.z2)
        print("After second layer: z2 shape:", self.z2.shape, "a2 shape:", self.a2.shape)

        self.z3 = self.a2 @ self.W3 + self.b3
        print("Final output: z3 shape:", self.z3.shape)
        return self.z3

    def backward(self, X, y, y_pred):
        """Backpropagation."""
        m = len(y)

        print("\nBackward pass:")
        dL_dy = mse_derivative(y, y_pred)

        dW3 = self.a2.T @ dL_dy
        db3 = np.sum(dL_dy, axis=0, keepdims=True)
        print("dW3 shape:", dW3.shape, "db3 shape:", db3.shape)

        da2 = dL_dy @ self.W3.T
        dz2 = da2 * relu_derivative(self.z2)
        print("dz2 shape:", dz2.shape)

        dW2 = self.a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)
        print("dW2 shape:", dW2.shape, "db2 shape:", db2.shape)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * relu_derivative(self.z1)
        print("dz1 shape:", dz1.shape)

        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)
        print("dW1 shape:", dW1.shape, "db1 shape:", db1.shape)

        # Gradient descent update
        print("\nUpdating weights and biases...")
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3
        print("W3 shape:", self.W3.shape, "b3 shape:", self.b3.shape)

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        print("W2 shape:", self.W2.shape, "b2 shape:", self.b2.shape)

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        print("W1 shape:", self.W1.shape, "b1 shape:", self.b1.shape)

    def fit(self, X, y, epochs=500):
        """Training loop."""
        losses = []

        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = mse(y, y_pred)
            self.backward(X, y, y_pred)
            losses.append(loss)

            if epoch % 50 == 0:
                print(f"Epoch {epoch} | Loss: {loss:.4f}")

        return losses

    def predict(self, X):
        """Inference."""
        return self.forward(X)


In [24]:
# 4.4 – Train the Model
# Convert to numpy
X_train_np = X_train_final.values
X_val_np   = X_val_final.values
X_test_np  = X_test_final.values

y_train_np = y_train.values.reshape(-1, 1)
y_val_np   = y_val.values.reshape(-1, 1)
y_test_np  = y_test.values.reshape(-1, 1)

# Initialize model
mlp = MLPRegressor(input_dim=X_train_np.shape[1])

# Train
losses = mlp.fit(X_train_np, y_train_np, epochs=500)


Initialized W1 with shape: (10, 64)
Initialized b1 with shape: (1, 64)

Initialized W2 with shape: (64, 32)
Initialized b2 with shape: (1, 32)

Initialized W3 with shape: (32, 1)
Initialized b3 with shape: (1, 1)

After first layer: z1 shape: (852, 64) a1 shape: (852, 64)
After second layer: z2 shape: (852, 32) a2 shape: (852, 32)
Final output: z3 shape: (852, 1)

Backward pass:
dW3 shape: (32, 1) db3 shape: (1, 1)
dz2 shape: (852, 32)
dW2 shape: (64, 32) db2 shape: (1, 32)
dz1 shape: (852, 64)
dW1 shape: (10, 64) db1 shape: (1, 64)

Updating weights and biases...
W3 shape: (32, 1) b3 shape: (1, 1)
W2 shape: (64, 32) b2 shape: (1, 32)
W1 shape: (10, 64) b1 shape: (1, 64)
Epoch 0 | Loss: nan

After first layer: z1 shape: (852, 64) a1 shape: (852, 64)
After second layer: z2 shape: (852, 32) a2 shape: (852, 32)
Final output: z3 shape: (852, 1)

Backward pass:
dW3 shape: (32, 1) db3 shape: (1, 1)
dz2 shape: (852, 32)
dW2 shape: (64, 32) db2 shape: (1, 32)
dz1 shape: (852, 64)
dW1 shape: (1

In [25]:
# 4.5 – Evaluate the Model
# Predictions
y_val_pred = mlp.predict(X_val_np)
y_test_pred = mlp.predict(X_test_np)

# Metrics
val_mse = mse(y_val_np, y_val_pred)
test_mse = mse(y_test_np, y_test_pred)

print("Validation MSE:", val_mse)
print("Test MSE:", test_mse)



After first layer: z1 shape: (182, 64) a1 shape: (182, 64)
After second layer: z2 shape: (182, 32) a2 shape: (182, 32)
Final output: z3 shape: (182, 1)

After first layer: z1 shape: (184, 64) a1 shape: (184, 64)
After second layer: z2 shape: (184, 32) a2 shape: (184, 32)
Final output: z3 shape: (184, 1)
Validation MSE: nan
Test MSE: nan


In [26]:
print("NaNs in X:", np.isnan(X_train_np).sum())
print("NaNs in y:", np.isnan(y_train_np).sum())
print("Max X:", np.max(X_train_np))
print("Min X:", np.min(X_train_np))


NaNs in X: 0
NaNs in y: 1
Max X: 18000000.0
Min X: -2.476980020357006


In [27]:
# Re-run the Entire Pipeline (clean version)
# 1. Parse target
y_raw = parse_price_column(df["Cars Prices"])
X_raw = df.drop(columns=["Cars Prices"])

# 2. Drop invalid targets
mask = y_raw.notna()
X_raw = X_raw[mask]
y_raw = y_raw[mask]

# 3. Split
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(X_raw, y_raw)

# 4. Identify types
categorical_cols, numerical_cols = identify_feature_types(X_train_raw)

# 5. Fix missing values
for col in numerical_cols:
    mean = X_train_raw[col].mean()
    X_train_raw[col] = X_train_raw[col].fillna(mean)
    X_val_raw[col]   = X_val_raw[col].fillna(mean)
    X_test_raw[col]  = X_test_raw[col].fillna(mean)

for col in categorical_cols:
    X_train_raw[col] = X_train_raw[col].fillna("UNKNOWN")
    X_val_raw[col]   = X_val_raw[col].fillna("UNKNOWN")
    X_test_raw[col]  = X_test_raw[col].fillna("UNKNOWN")

# 6. Target encoding
X_train_te, X_val_te, X_test_te, te_maps = target_encode(
    X_train_raw, X_val_raw, X_test_raw, y_train, categorical_cols
)

# 7. Standardization
X_train_final, X_val_final, X_test_final, scaling_stats = standardize_features(
    X_train_te, X_val_te, X_test_te, numerical_cols
)

# 8. Final numpy
X_train_np = X_train_final.values
y_train_np = y_train.values.reshape(-1,1)


Encoded Company Names with 35 categories.
Global mean for unseen categories: 160380.74
Encoded Cars Names with 844 categories.
Global mean for unseen categories: 160380.74
Encoded Engines with 288 categories.
Global mean for unseen categories: 160380.74
Encoded CC/Battery Capacity with 268 categories.
Global mean for unseen categories: 160380.74
Encoded HorsePower with 366 categories.
Global mean for unseen categories: 160380.74
Encoded Total Speed with 92 categories.
Global mean for unseen categories: 160380.74
Encoded Performance(0 - 100 )KM/H with 157 categories.
Global mean for unseen categories: 160380.74
Encoded Fuel Types with 18 categories.
Global mean for unseen categories: 160380.74
Encoded Torque with 218 categories.
Global mean for unseen categories: 160380.74
Standardization stats:
  Seats: mean=4.86, std=1.56
X_train shape: (851, 10)


In [28]:
# 4.5 – Evaluate the Model
# Predictions
y_val_pred = mlp.predict(X_val_np)
y_test_pred = mlp.predict(X_test_np)

# Metrics
val_mse = mse(y_val_np, y_val_pred)
test_mse = mse(y_test_np, y_test_pred)

print("Validation MSE:", val_mse)
print("Test MSE:", test_mse)



After first layer: z1 shape: (182, 64) a1 shape: (182, 64)
After second layer: z2 shape: (182, 32) a2 shape: (182, 32)
Final output: z3 shape: (182, 1)

After first layer: z1 shape: (184, 64) a1 shape: (184, 64)
After second layer: z2 shape: (184, 32) a2 shape: (184, 32)
Final output: z3 shape: (184, 1)
Validation MSE: nan
Test MSE: nan
